# Stage 5 — Grounding Rate Only
Runs grounding rate for Stage 3 (hybrid+rerank) and Stage 4b (HyDE+rerank).
No Recall/MRR loop — won't burn tokens on HyDE passages before grounding starts.

In [1]:
# Cell 1 — Clone repo and install deps
!git clone https://github.com/HarshAggarwal524/sourcerer.git
%cd sourcerer


fatal: destination path 'sourcerer' already exists and is not an empty directory.
/content/sourcerer


In [2]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 104.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.6.0
    Uninstalling sentence-transformers-5.6.0:
      Successfully uninstalled sentence-transformers-5.6.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following depe

In [2]:
# Cell 2 — Set your Groq API key
import os
os.environ['GROQ_API_KEY'] = 'YOUR_API_KEY'

In [6]:
script = '''
import sys
sys.path.insert(0, "/content/sourcerer")

from main import load_or_build
from eval.retrieve import hybrid_rerank_retrieve, hyde_rerank_retrieve
from eval.generate import generate_answer
from eval.stage5_trust import check_grounding
import json

def run_grounding(test_path, chunks, embeddings, bm25_index, mode, label):
    with open(test_path) as f:
        data = json.load(f)

    supported = 0
    total = 0

    for item in data:
        question = item["question"]

        if mode == "hybrid_rerank":
            top_chunks = hybrid_rerank_retrieve(question, chunks, embeddings, bm25_index, top_k=1)
        else:
            top_chunks = hyde_rerank_retrieve(question, chunks, embeddings, bm25_index, top_k=1)

        if not top_chunks:
            total += 1
            continue

        chunk_text = top_chunks[0]
        answer = generate_answer(question, chunk_text)

        if answer:
            result = check_grounding(question, chunk_text, answer)
            if result == "SUPPORTED":
                supported += 1
        total += 1

    grounding = supported / total if total > 0 else 0
    print(f"{label}")
    print(f"Grounding: {grounding:.4f}  ({supported}/{total})\\n")
    return grounding

chunks, embeddings, bm25_index = load_or_build()

testsets = [
    ("eval/easy.json",      "Easy"),
    ("eval/hard.json",      "Hard"),
    ("eval/hardest.json",   "Hardest"),
    ("eval/ambiguous.json", "Ambiguous"),
]

results = []

print("--- Stage 3 (hybrid + rerank, no HyDE) ---\\n")
for path, set_label in testsets:
    g = run_grounding(path, chunks, embeddings, bm25_index, "hybrid_rerank", f"{set_label} — Stage 3")
    results.append((f"{set_label} — Stage 3", g))

print("--- Stage 4b (HyDE + rerank) ---\\n")
for path, set_label in testsets:
    g = run_grounding(path, chunks, embeddings, bm25_index, "hyde_rerank", f"{set_label} — Stage 4b")
    results.append((f"{set_label} — Stage 4b", g))

print("========== STAGE 5 SUMMARY ==========")
print(f"{'Label':<45} {'Grounding'}")
for label, g in results:
    print(f"{label:<45} {g:.4f}")
'''

with open('/content/sourcerer/run_grounding.py', 'w') as f:
    f.write(script)
print('Script written.')

Script written.


In [8]:
! git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 359 bytes | 359.00 KiB/s, done.
From https://github.com/HarshAggarwal524/sourcerer
   47c6c5a..59a8660  main       -> origin/main
Updating 47c6c5a..59a8660
Fast-forward
 eval/score.py | 4 ++--
 1 file changed, 2 insertions(+), 2 deletions(-)


In [9]:
!PYTHONPATH=/content/sourcerer python eval/score.py

Loading weights: 100% 103/103 [00:00<00:00, 5188.23it/s]
Loading weights: 100% 393/393 [00:00<00:00, 7012.28it/s]
Found existing cache: cache/store_63646eb77a100482.pkl
Loaded 33 chunks.

========== STAGE 5: Grounding Rate ==========

--- Stage 3 pipeline (hybrid + rerank, no HyDE) ---

--- Easy — Stage 3 (Grounding) (30 questions) ---
Recall@3:  1.0000
MRR:       1.0000
Grounding: 0.9667

--- Hard — Stage 3 (Grounding) (29 questions) ---
Recall@3:  0.9310
MRR:       0.9224
Grounding: 0.6897

--- Hardest — Stage 3 (Grounding) (22 questions) ---
Recall@3:  0.8636
MRR:       0.7765
Grounding: 0.3636

--- Ambiguous — Stage 3 (Grounding) (29 questions) ---
Recall@3:  0.8621
MRR:       0.6968
Grounding: 0.3793

--- Stage 4b pipeline (HyDE + rerank) ---

[HyDE] Original question: What was the name of the cooperative community that Robert Owen sought to build in Indiana, USA?
[HyDE] Hypothetical passage: In 1825, Robert Owen, a British industrialist and social reformer, established a cooperat